In Gradient Boosted Decision Trees (GBDT), additive training means building the model sequentially, one tree at a time, where each new tree is trained specifically to fix the remaining errors (residuals) of all previous trees combined.



since we all here are familiar about tree boostings , lets dig deeper into it 


we know reqularized learning objective ,

$$ y(i)_(pred) = \phi(x_i) =  \sum_{k=1}^{k} f_i(x_i) $$

to minimize the set of functions, we introduce regularization penalty to the system whic is responsible for overfitting of data

$$L(\phi) = \sum_{i}^{} l(y_ipred, y_i) + \sum_{i}^{} \Omega(f_k)$$

where,

$$\Omega(f_k) = \gamma T + 1/2 \lambda||\omega||^2$$

here teh fuunctions and the parameters cannot be optimized using the traditional optimization methods in eucledian space.In traditional continuous models (like linear regression or neural networks), your model parameters live in a continuous Euclidean vector space ($\mathbb{R}^d$). You can smoothly adjust weights using gradients because the loss surface with respect to those parameters is continuous and differentiable.

You can't navigate the parameter space of a decision tree via continuous Euclidean vectors because tree structures are discrete. GBDT circumvents this by calculating continuous gradients on the output predictions instead, then using a greedy search to fit a tree to those target gradients.



$$ obj_t = L_t = \sum_{i}^{} l(y_ipred, y_i) + f_t(x_i) + \Omega(f_k) $$ $$ 

here we add the function ($f_t(x_i)$)greedily meaning we optimize and fix $f_t$ without ever going back to re-adjust or jointly optimize previous trees—for one main reason: joint optimization across all trees simultaneously is computationally impossible. It turns an intractable NP-complete joint search into a sequence of small, highly efficient single-tree fits.

If we didn't train greedily, every time we wanted to add a tree, we would have to solve for the split decisions and leaf values of all $t$ trees at once.

Greedy stage-wise additive modeling reduces an impossible $100$-tree joint optimization problem into 100 manageable single-tree optimization problems, solved sequentially.

now first order optimization would tell us the direction of error. It treats flat regions and steep regions the same whereas the second order approximation measures the slope of the curvature

This is Newton-Raphson optimization applied at the leaf level, causing boosting to converge in significantly fewer trees.

TAILOR SERIES EXPANSION

$$f(x) \approx f(a) + \frac{1}{1!} f'(a)(x - a) + \frac{1}{2!}f''(a)(x - a)^2 + ...$$

$$ => f(x) \approx f(a) + f'(a)(x - a) + \frac{1}{2}f''(a)(x - a)^2$$

so,

$$=> L\left(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\right) \approx L\left(y_i, \hat{y}_i^{(t-1)}\right) + g_i f_t(x_i) + \frac{1}{2} h_i f_t(x_i)^2$$

where gi and hi are gradients and hessians respectively,

since $L\left(y_i, \hat{y}_i^{(t-1)}\right)$ is a constant so we can neglect it


$$\tilde{\mathcal{L}}^{(t)} \approx \sum_{i=1}^n \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t(x_i)^2 \right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2$$

so let $ I_j = ({i, q(x_i) = j})$
when we simplify it we get
$f_t(x_i) = w_i$

so,

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^T \left[ \left( \sum_{i \in I_j} g_i \right) w_j + \frac{1}{2} \left( \sum_{i \in I_j} h_i + \lambda \right) w_j^2 \right] + \gamma T$$

Let $G_j = \sum_{i \in I_j} g_i$ (sum of first derivatives in leaf $j$) and $H_j = \sum_{i \in I_j} h_i$ (sum of second derivatives in leaf $j$):

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^T \left[ G_j w_j + \frac{1}{2} (H_j + \lambda) w_j^2 \right] + \gamma T$$

Derived by setting $\frac{\partial \tilde{\mathcal{L}}^{(t)}}{\partial w_j} = 0$:

$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

Substituting $w_j^*$ back into the objective function:

$$\tilde{\mathcal{L}}^{(t)} = -\frac{1}{2} \sum_{j=1}^T \frac{G_j^2}{H_j + \lambda} + \gamma T$$

The score used to decide whether to split a node into Left ($L$) and Right ($R$) subnodes:

$$\text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma$$

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^T \left[ G_j w_j + \frac{1}{2} (H_j + \lambda) w_j^2 \right] + \gamma T$$

where:
* $G_j = \sum_{i \in I_j} g_i$ is the sum of first-order gradients in leaf $j$.
* $H_j = \sum_{i \in I_j} h_i$ is the sum of second-order gradients (Hessians) in leaf $j$.
* $w_j$ is the continuous weight score assigned to leaf $j$.
* $\lambda$ is the $L_2$ regularization parameter.
* $T$ is the total number of leaves, and $\gamma$ is the tree complexity penalty.

Since the leaf structure partitions the data instances into disjoint sets ($I_j \cap I_k = \emptyset$ for $j \neq k$), each leaf weight $w_j$ can be optimized independently. 

For a single fixed leaf $j$, the loss contribution dependent on $w_j$ is a quadratic function $f(w_j)$:

$$f(w_j) = G_j w_j + \frac{1}{2} (H_j + \lambda) w_j^2$$

To find the weight $w_j$ that minimizes $f(w_j)$, take the partial derivative with respect to $w_j$ and set it to zero:

$$\frac{\partial \tilde{\mathcal{L}}^{(t)}}{\partial w_j} = \frac{\partial}{\partial w_j} \left[ G_j w_j + \frac{1}{2} (H_j + \lambda) w_j^2 \right] = 0$$

Differentiating term-by-term:

$$G_j + (H_j + \lambda) w_j = 0$$

Rearranging the terms:

$$(H_j + \lambda) w_j = -G_j$$

$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

Taking the second derivative with respect to $w_j$:

$$\frac{\partial^2 \tilde{\mathcal{L}}^{(t)}}{\partial w_j^2} = H_j + \lambda$$

Since $h_i > 0$ for typical convex loss functions and $\lambda \ge 0$, we have $H_j + \lambda > 0$. Because the second derivative is strictly positive, $w_j^*$ guarantees a global **minimum** for leaf $j$

## Derivation of Node Quality Score (Minimal Objective)

### 1. Objective Function in Compact Form

Recall the second-order Taylor approximation of the objective function simplified using aggregated gradients:

$$\tilde{\mathcal{L}}^{(t)} = \sum_{j=1}^T \left[ G_j w_j + \frac{1}{2} (H_j + \lambda) w_j^2 \right] + \gamma T$$

where:
* $G_j = \sum_{i \in I_j} g_i$ (sum of first-order gradients in leaf $j$)
* $H_j = \sum_{i \in I_j} h_i$ (sum of second-order gradients in leaf $j$)
* $w_j$ is the continuous prediction score for leaf $j$
* $\lambda$ is the $L_2$ regularization parameter

---

### 2. Optimal Leaf Weight Substitution

The optimal weight $w_j^*$ that minimizes the objective for each leaf $j$ is:

$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

Substituting $w_j^*$ into the loss contribution function $f(w_j) = G_j w_j + \frac{1}{2} (H_j + \lambda) w_j^2$ for a single leaf $j$:

$$f(w_j^*) = G_j \left( -\frac{G_j}{H_j + \lambda} \right) + \frac{1}{2} (H_j + \lambda) \left( -\frac{G_j}{H_j + \lambda} \right)^2$$

---

### 3. Simplifying the Leaf Loss Expression

1. **Expand the first term:**
   $$-\frac{G_j^2}{H_j + \lambda}$$

2. **Expand the second term:**
   $$\frac{1}{2} (H_j + \lambda) \cdot \frac{G_j^2}{(H_j + \lambda)^2} = \frac{1}{2} \frac{G_j^2}{H_j + \lambda}$$

3. **Combine both terms:**
   $$f(w_j^*) = -\frac{G_j^2}{H_j + \lambda} + \frac{1}{2} \frac{G_j^2}{H_j + \lambda} = -\frac{1}{2} \frac{G_j^2}{H_j + \lambda}$$

---

### 4. Minimal Total Objective Function

Summing $f(w_j^*)$ across all $T$ leaves and re-adding the tree regularization penalty $\gamma T$ yields the overall minimal loss value:

$$\tilde{\mathcal{L}}^{(t)*} = -\frac{1}{2} \sum_{j=1}^T \frac{G_j^2}{H_j + \lambda} + \gamma T$$

---

### 5. Node Quality Definition

For any single node $j$ (whether a leaf or an internal candidate node), the quantity:

$$\text{Score}(j) = \frac{G_j^2}{H_j + \lambda}$$

serves as the **Node Quality Score** (or similarity score). 

* Higher values indicate a larger reduction in training loss.
* $\lambda$ acts as a denominator penalty, dampening nodes with very small Hessians or low data cover ($H_j$).

## Derivation of Split Criterion (Gain)

### 1. Structural Quality Score of a Node

From the optimal objective value, the minimal loss contribution of a node $j$ (ignoring the tree penalty $\gamma T$) depends solely on the aggregated first and second gradients:

$$\text{Score}(j) = \frac{1}{2} \frac{G_j^2}{H_j + \lambda}$$

This score measures the quality of a leaf structure: a larger score indicates a greater reduction in overall loss.

---

### 2. Evaluating a Potential Split

Suppose a parent node $P$ containing dataset $I$ is considered for a split into a left child node $L$ ($I_L$) and a right child node $R$ ($I_R$), such that $I = I_L \cup I_R$ and $I_L \cap I_R = \emptyset$.

By gradient additivity:
* $G_P = G_L + G_R = \sum_{i \in I_L} g_i + \sum_{i \in I_R} g_i$
* $H_P = H_L + H_R = \sum_{i \in I_L} h_i + \sum_{i \in I_R} h_i$

---

### 3. Comparing Loss Before and After Split

* **Loss before split (single leaf $P$):**
  $$\tilde{\mathcal{L}}_{\text{before}} = -\frac{1}{2} \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} + \gamma$$

* **Loss after split (two leaves $L$ and $R$):**
  $$\tilde{\mathcal{L}}_{\text{after}} = -\frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} \right] + 2\gamma$$

---

### 4. Calculating the Gain

The split gain is defined as the reduction in loss after splitting ($\text{Gain} = \tilde{\mathcal{L}}_{\text{before}} - \tilde{\mathcal{L}}_{\text{after}}$):

$$\text{Gain} = \left( -\frac{1}{2} \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} + \gamma \right) - \left( -\frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} \right] + 2\gamma \right)$$

Simplifying terms yields the final XGBoost split gain formula:

$$\text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_R + H_L + \lambda} \right] - \gamma$$

---

### 5. Interpretation of the Gain Metric

* **$\frac{G_L^2}{H_L + \lambda}$**: Score contribution of the left child.
* **$\frac{G_R^2}{H_R + \lambda}$**: Score contribution of the right child.
* **$\frac{(G_L + G_R)^2}{H_L + H_R + \lambda}$**: Score contribution if the node is not split.
* **$\gamma$**: The complexity threshold (regularization term). If the raw gain is less than $\gamma$, the total gain becomes negative, and the split is pruned.